# End-to-End Demo (Team DataX)

Walks through the full Bronze → Silver → POI → Gold → Model pipeline plus
the Round-2 business-decision modules and the web-app deliverables.

Sections:
1. Data loading & schema validation
2. EDA snapshots (distributions, missingness, before/after dedup)
3. Pipeline run (12 phases via `src/run_pipeline.py`)
4. Predictions sanity check
5. Result visualisation
6. Round-2 modules — cooler ROI, action cards, dormancy risk, distributor scorecard,
   HDBSCAN territories, SHAP attribution, counterfactual deltas
7. Web app entry points (FastAPI on :8000, Next.js on :3000)

The companion documents are `README.md` (setup) and `docs/methodology_overview.md`
(full per-module reference with diagrams).

## 1. Data loading & schema validation

In [ ]:
import sys
from pathlib import Path
if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

print('PROJECT_ROOT:', config.PROJECT_ROOT)
print('RAW_SOURCE  :', config.RAW_SOURCE, '(exists:', config.RAW_SOURCE.exists(), ')')
print('TEAM_NAME   :', config.TEAM_NAME)

In [ ]:
# Inspect the raw schemas (uses cached bronze if already populated)
src = config.BRONZE if (config.BRONZE / 'outlet_master.csv').exists() else config.RAW_SOURCE

files = {
    'transactions': 'transactions_history_final.csv',
    'master'      : 'outlet_master.csv',
    'coords'      : 'outlet_coordinates.csv',
    'seasonality' : 'distributor_seasonality_details.csv',
    'holidays'    : 'holiday_list.csv',
}
schema = []
for label, fname in files.items():
    df = pd.read_csv(src / fname, nrows=2)
    n = sum(1 for _ in open(src / fname, 'rb')) - 1
    schema.append({'dataset': label, 'rows': n, 'columns': list(df.columns)})
pd.DataFrame(schema)

## 2. EDA snapshots

Selected views â€” distributions, missingness, before/after dedup. Heavy lifting is in `src/silver_clean.py`; this is just the orientation.

In [ ]:
txns = pd.read_csv(src / 'transactions_history_final.csv')
print('Shape:', txns.shape)
print('Year/Month range:')
print(txns.groupby('Year')['Month'].agg(['min', 'max']))

In [ ]:
# Volume_Liters distribution (clipped at 99th percentile for legibility)
q99 = txns['Volume_Liters'].quantile(0.99)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(txns['Volume_Liters'].clip(-50, q99), bins=60, color='steelblue', edgecolor='white')
ax[0].set_title(f'Volume_Liters per SKU-month (clipped at 99th pct = {q99:.1f})')
ax[0].set_xlabel('Volume_Liters')

txns['Outlet_Year_Month'] = txns['Outlet_ID'] + '|' + txns['Year'].astype(str) + '|' + txns['Month'].astype(str)
skus_per_om = txns.groupby('Outlet_Year_Month').size()
ax[1].hist(skus_per_om, bins=range(1, 12), color='seagreen', edgecolor='white')
ax[1].set_title('SKU lines per outlet-month')
ax[1].set_xlabel('Distinct SKU lines')
plt.tight_layout()

In [ ]:
# Outlet_Type before vs after typo normalisation
master = pd.read_csv(src / 'outlet_master.csv')
print('Outlet_Type value counts (raw):')
print(master['Outlet_Type'].value_counts(dropna=False).head(15))

if (config.SILVER_CLEAN / 'outlet_master_clean.parquet').exists():
    cleaned = pd.read_parquet(config.SILVER_CLEAN / 'outlet_master_clean.parquet')
    print('\nOutlet_Type value counts (cleaned):')
    print(cleaned['Outlet_Type'].value_counts(dropna=False).head(15))

## 3. Pipeline run

The next cell is destructive in the sense that it overwrites `data/silver/`, `data/gold/`, and `outputs/`. It is what `python -m src.run_pipeline` does outside the notebook. Skip if you have already run the CLI orchestrator.

In [ ]:
# Skip if you've already run python -m src.run_pipeline
RUN_PIPELINE = False
if RUN_PIPELINE:
    from src import bronze_ingest, silver_clean, poi_scraper, gold_features, potential_model
    bronze_ingest.main()
    silver_clean.main()
    poi_scraper.main()
    gold_features.main()
    potential_model.main()

## 4. Predictions sanity check

In [ ]:
pred = pd.read_csv(config.PREDICTIONS_CSV)
print('Rows:', len(pred))
print('Columns:', list(pred.columns))
print('Distinct Outlet_IDs:', pred['Outlet_ID'].nunique())
print('Nulls:', pred.isna().sum().sum())
print('Non-positive:', (pred['Maximum_Monthly_Liters'] <= 0).sum())
pred['Maximum_Monthly_Liters'].describe().round(2)

In [ ]:
# Predicted vs observed-mean ratio per outlet
feats = pd.read_parquet(config.GOLD / 'outlet_features.parquet')
merge = pred.merge(
    feats[['Outlet_ID', 'monthly_volume_mean', 'Province', 'Outlet_Type']],
    on='Outlet_ID', how='left'
)
merge['ratio'] = merge['Maximum_Monthly_Liters'] / merge['monthly_volume_mean'].replace(0, np.nan)
print('Median predicted / observed-mean ratio:', merge['ratio'].median().round(2))
print('Per-province summary of predicted potential:')
merge.groupby('Province')['Maximum_Monthly_Liters'].agg(['count', 'median', 'mean', 'max']).round(2)

## 5. Result visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.log1p(pred['Maximum_Monthly_Liters']), bins=50, color='steelblue', edgecolor='white')
ax[0].set_title('Predicted potential (log scale)')
ax[0].set_xlabel('log(1 + Maximum_Monthly_Liters)')

by_prov = merge.dropna(subset=['Province']).groupby('Province')['Maximum_Monthly_Liters'].median()
by_prov.plot(kind='bar', ax=ax[1], color='seagreen')
ax[1].set_title('Median predicted potential by province')
ax[1].set_ylabel('Liters / month')
plt.tight_layout()

In [ ]:
# Geographic scatter â€” prediction magnitude by location
coords = pd.read_parquet(config.SILVER_CLEAN / 'outlet_coordinates_clean.parquet')
geo = pred.merge(coords, on='Outlet_ID', how='inner')
fig, ax = plt.subplots(figsize=(8, 8))
sc = ax.scatter(geo['Longitude'], geo['Latitude'],
                c=np.log1p(geo['Maximum_Monthly_Liters']),
                s=2, cmap='viridis')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Predicted potential across Sri Lanka (log scale)')
plt.colorbar(sc, label='log(1 + Maximum_Monthly_Liters)')
plt.tight_layout()

## 6. Round-2 modules

The pipeline now also produces five business-decision artifacts. They depend on
the predictions + features the cells above already generated, so re-running
`src.run_pipeline` is enough.

| Module | Output |
|---|---|
| Cooler ROI | `outputs/audit/cooler_roi_top100.csv` |
| Outlet action cards | `data/gold/outlet_actions.parquet` + `outputs/audit/outlet_actions_top3.csv` |
| Dormancy classifier | `data/gold/dormancy_risk.parquet` + `outputs/audit/dormancy_top200_at_risk.csv` |
| Distributor scorecard | `outputs/audit/distributor_scorecard.csv` |
| HDBSCAN territories | `data/gold/outlet_clusters.parquet` + `outputs/audit/territory_clusters_summary.csv` |
| SHAP attribution | `data/gold/shap_values.parquet` + `outputs/audit/shap_top_drivers_per_outlet.csv` |
| Counterfactuals | `data/gold/counterfactuals.parquet` |

Quick read of the cooler ROI top-line so this notebook reflects the
Round-2 business story end to end.

In [ ]:
import pandas as pd
roi = pd.read_csv(config.AUDIT / 'cooler_roi_top100.csv')
cooler_unit_cost = 50_000
capex = len(roi) * cooler_unit_cost
margin_24mo = roi['monthly_margin_uplift_LKR'].sum() * 24
print(f"Top-100 cooler deployment:")
print(f"  Capex required (LKR):        {capex:,.0f}")
print(f"  24-month margin uplift (LKR):{margin_24mo:>20,.0f}")
print(f"  Net 24-month value (LKR):    {margin_24mo - capex:>20,.0f}")
print(f"  Median payback (months):     {roi['payback_months'].median():.1f}")
print(f"  Outlets in Top-100:          {len(roi)}")

## 7. Web app

Two-process app that surfaces every output above to a business user:

```bash
# Terminal 1
cd webapp/backend
pip install -r requirements.txt
cp .env.example .env   # add GEMINI_API_KEY
python -m uvicorn main:app --port 8000

# Terminal 2
cd webapp/frontend
npm install
npm run dev   # http://localhost:3000
```

Pages:
- `/` Dashboard — KPI tiles, province breakdown
- `/outlets` 20K outlets, 6 filters, server-side sort, 20 per page
- `/outlets/{id}` SHAP drivers, counterfactual deltas, recommended actions, cooler ROI snapshot, Gemini AI narrative
- `/insights?view={budget,cooler-roi,dormancy,scorecard,territories,forensics}` Tabbed analytical workspace, all paginated at 20 per page

The Gemini call only fires when the user clicks "Explain this outlet" — we do not
pre-generate narratives for all 20,000 outlets.